# OpenBind-HIPPO

- **Target: TARGET_NAME**
- **Cycle: CYCLE_NUMBER**

## Prep

- [ ] Downloaded TARGET_NAME with fragalysis_download.ipynb
- [ ] Bulkdock setup: `python -m bulkdock setup TARGET_NAME`
- [ ] Copy aligned files to here: `cp -rv $BULK/TARGETS/TARGET_NAME/aligned_files .`

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp

# Config

In [ ]:
target_name = "TARGET_NAME"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_CYCLE_NUMBER"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [ ]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

## Merging

Run merging algorithms on all poses tagged 'hits'

In [ ]:
merge_input_poses = animal.poses(tag="hits")
merge_input_poses

### Create inputs

- CSV input for Knitwork
- SDF of hits to merge for Knitwork and Fragmenstein
- Protonated PDBs for Knitwork
- Reference PDB for Fragmenstein

In [ ]:
# output directories
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_out_dir = cycle_dir / "fragmenstein"
fragmenstein_out_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# knitwork CSV
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
merge_input_poses.to_knitwork(knitwork_out_dir / f"{cycle_name}_input.csv", path_root=knitwork_out_dir, aligned_files_dir="aligned_files")

In [ ]:
# reference apo PDB for Fragmenstein
ref_pose = merge_input_poses[0]
mrich.var("ref_pose", ref_pose)
shutil.copy(ref_pose.apo_path, fragmenstein_out_dir)

In [ ]:
# SDF of hits
out_dir = cycle_dir
out_dir.mkdir(parents=True, exist_ok=True)
merge_input_poses.write_sdf(cycle_dir / f"{cycle_name}_hits.sdf")

In [ ]:
#protonated PDBs
for pose in mrich.track(merge_input_poses):
    path = pose.apo_path
    sys = mp.parse(path,verbosity=False)
    sys.add_hydrogens(pH=7)
    subdir = Path("aligned_files") / pose.name
    subdir.mkdir(exist_ok=True, parents=True)
    sys.write(subdir / str(path.name).replace('_apo-desolv.pdb', '_apo-desolv-Hs.pdb'), verbosity=1)

### Run Fragmenstein

```
cd cycle_CYCLE_NUMBER/fragmenstein
sbatch --job-name "TARGET_NAME_fragmenstein" --mem 16000 $HOME2/slurm/run_bash_with_conda.sh ../../../scripts/run_fragmenstein.sh
```

### Run Knitwork

- [ ] Clone repo on graph-sw-2
- [ ] `cd TARGET_NAME/cycle_CYCLE_NUMBER/knitwork`
- [ ] Run the "fragment" step of FragmentKnitwork: `../../../scripts/run_knitwork_fragment.sh`
- [ ] Run the pure "knitting" step of FragmentKnitwork:`../../../scripts/run_knitwork_pure.sh`
- [ ] Run the impure "knitting" step of FragmentKnitwork: `../../../scripts/run_knitwork_impure.sh`
- [ ] Create bulkdock inputs: `knitwork_to_bulkdock.py`